# 🏋️ Challenge 05 — Exceptions, File Handling, CSV & JSON
*Covers Modules 11–12.* 🟢 easy · 🟡 medium · 🔴 hard. The tests create their own temporary files — you don't need any setup.

In [ ]:
import os, json, csv, tempfile

def check(name, test):
    try:
        ok = bool(test())
    except Exception as e:
        ok = False
        name += f"   (raised {type(e).__name__}: {e})"
    print("✅ PASS" if ok else "❌ FAIL", "-", name)

TMP = tempfile.mkdtemp()          # scratch folder for this notebook
def path(name): return os.path.join(TMP, name)

### 🟢 Problem 1 — Safe division
`safe_divide(a, b)` returns `a / b`. If `b` is zero return the string `"cannot divide by zero"`. If either argument is not a number return `"invalid input"`. Use `try/except` (no `if b == 0` check).

In [ ]:
def safe_divide(a, b):
    pass

In [ ]:
check("normal", lambda: safe_divide(10, 4) == 2.5)
check("zero", lambda: safe_divide(1, 0) == "cannot divide by zero")
check("bad type", lambda: safe_divide("a", 2) == "invalid input")

### 🟢 Problem 2 — Parse integers safely
`parse_ints(tokens)` takes a list of strings and returns `(numbers, bad)`: the list of successfully converted ints and the list of tokens that failed. Whitespace like `" 7 "` is fine (int() allows it).

In [ ]:
def parse_ints(tokens):
    pass

In [ ]:
check("mixed", lambda: parse_ints(["1", "x", " 7 ", "3.5", "-2"]) == ([1, 7, -2], ["x", "3.5"]))
check("empty", lambda: parse_ints([]) == ([], []))

### 🟢 Problem 3 — Write and read a file
`write_lines(filename, lines)` writes each item of `lines` on its own line (add `"\n"`). `read_lines(filename)` returns the list of lines **without** the newline characters. Use `with open(...)`.

In [ ]:
def write_lines(filename, lines):
    pass

def read_lines(filename):
    pass

In [ ]:
check("round trip", lambda: (write_lines(path("a.txt"), ["one", "two", "three"]), read_lines(path("a.txt")))[1] == ["one", "two", "three"])
check("overwrite", lambda: (write_lines(path("a.txt"), ["x"]), read_lines(path("a.txt")))[1] == ["x"])
check("empty file", lambda: (write_lines(path("e.txt"), []), read_lines(path("e.txt")))[1] == [])

### 🟢 Problem 4 — Append & count
`append_line(filename, text)` appends a line (creating the file if needed). `count_lines_words_chars(filename)` returns `(lines, words, characters)` — characters exclude the newline characters.

In [ ]:
def append_line(filename, text):
    pass

def count_lines_words_chars(filename):
    pass

In [ ]:
def _t4():
    p = path("log.txt")
    if os.path.exists(p): os.remove(p)
    append_line(p, "hello world"); append_line(p, "python is fun")
    return count_lines_words_chars(p)
check("count", lambda: _t4() == (2, 5, 24))

### 🟡 Problem 5 — Missing file handling
`read_or_default(filename, default="")` returns the file's content, or `default` if the file doesn't exist. `read_config_value(filename, key)` reads a `key=value` file and returns the value for `key` (stripped), or `None` if the key or file is missing. Ignore lines without `=`.

In [ ]:
def read_or_default(filename, default=""):
    pass

def read_config_value(filename, key):
    pass

In [ ]:
open(path("cfg.txt"), "w").write("name = Asha\n# comment\ncity=Pune\nnoequals\n")
check("default when missing", lambda: read_or_default(path("nope.txt"), "N/A") == "N/A")
check("reads content", lambda: read_or_default(path("cfg.txt")).startswith("name"))
check("config value", lambda: read_config_value(path("cfg.txt"), "name") == "Asha")
check("config value 2", lambda: read_config_value(path("cfg.txt"), "city") == "Pune")
check("missing key", lambda: read_config_value(path("cfg.txt"), "age") is None)
check("missing file", lambda: read_config_value(path("nope.txt"), "x") is None)

### 🟡 Problem 6 — Custom exception
Create `class InsufficientFunds(Exception)` storing `balance` and `needed`, with a message like `"Need 50 more"`. Write `withdraw(balance, amount)` that returns the new balance or raises `InsufficientFunds`. Also raise `ValueError` if `amount <= 0`.

In [ ]:
class InsufficientFunds(Exception):
    pass

def withdraw(balance, amount):
    pass

In [ ]:
check("ok", lambda: withdraw(100, 30) == 70)
def _insufficient():
    try:
        withdraw(20, 70)
    except InsufficientFunds as e:
        return e.needed == 50 and e.balance == 20 and str(e) == "Need 50 more"
check("InsufficientFunds", _insufficient)
def _value():
    try:
        withdraw(20, 0)
    except ValueError:
        return True
check("ValueError for non-positive", _value)
check("is an Exception subclass", lambda: issubclass(InsufficientFunds, Exception))

### 🟡 Problem 7 — CSV: grade report
`average_by_student(filename)` reads a CSV with header `name,subject,marks` and returns `{name: average_marks_rounded_to_2}`. Skip rows whose marks aren't numbers.

In [ ]:
def average_by_student(filename):
    pass

In [ ]:
with open(path("marks.csv"), "w", newline="") as f:
    csv.writer(f).writerows([["name", "subject", "marks"], ["Asha", "Math", 90], ["Asha", "Sci", 85],
                             ["Ravi", "Math", 70], ["Ravi", "Sci", "absent"], ["Ravi", "Eng", 81]])
check("averages", lambda: average_by_student(path("marks.csv")) == {"Asha": 87.5, "Ravi": 75.5})

### 🟡 Problem 8 — JSON: save and load
`save_json(obj, filename)` and `load_json(filename, default=None)`. `load_json` returns `default` if the file is missing **or** contains invalid JSON.

<details><summary>💡 Hint</summary>Catch `FileNotFoundError` and `json.JSONDecodeError` (a subclass of `ValueError`).</details>

In [ ]:
def save_json(obj, filename):
    pass

def load_json(filename, default=None):
    pass

In [ ]:
data = {"name": "Asha", "scores": [1, 2, 3], "ok": True, "none": None}
check("round trip", lambda: (save_json(data, path("d.json")), load_json(path("d.json")))[1] == data)
check("missing file -> default", lambda: load_json(path("missing.json"), {}) == {})
open(path("bad.json"), "w").write("{not json")
check("invalid json -> default", lambda: load_json(path("bad.json"), "fallback") == "fallback")

### 🟡 Problem 9 — `finally` and return values
Write `process(value)` which:
1. tries `int(value)` and returns `n * 2`,
2. on `ValueError` returns `-1`,
3. **always** appends `"done"` to the module-level list `LOG` (use `finally`).

In [ ]:
LOG = []

def process(value):
    pass

In [ ]:
LOG.clear()
check("valid", lambda: process("21") == 42)
check("invalid", lambda: process("abc") == -1)
check("finally ran both times", lambda: LOG == ["done", "done"])

### 🔴 Problem 10 — Robust input loop
`read_valid_age(inputs)` simulates asking the user repeatedly. `inputs` is a list of strings (the "typed" answers, consumed in order). Return the first valid age: an integer between 1 and 120. Print nothing. If none is valid return `None`.

In [ ]:
def read_valid_age(inputs):
    pass

In [ ]:
check("first valid", lambda: read_valid_age(["abc", "-4", "200", "35", "40"]) == 35)
check("none valid", lambda: read_valid_age(["x", "0"]) is None)
check("boundaries", lambda: read_valid_age(["1"]) == 1 and read_valid_age(["120"]) == 120)

### 🔴 Problem 11 — Word frequency file report
`word_report(src, dst, top=3)` reads a text file, counts words (lowercase, strip `.,!?;:`), and writes to `dst` the `top` most common words, one per line as `word: count` (ties alphabetical). Return the list of lines written.

In [ ]:
def word_report(src, dst, top=3):
    pass

In [ ]:
open(path("story.txt"), "w").write("The cat sat. The dog sat! A cat, a dog, the end.")
expected = ["the: 3", "a: 2", "cat: 2"]
check("returns lines", lambda: word_report(path("story.txt"), path("report.txt")) == expected)
check("file written", lambda: open(path("report.txt")).read().splitlines() == expected)
check("top=1", lambda: word_report(path("story.txt"), path("r1.txt"), top=1) == ["the: 3"])

### 🔴 Problem 12 — Mini contact book (JSON persistence + errors)
Implement a class `ContactBook(filename)`:
- loads existing contacts from the JSON file (a dict `name -> phone`) or starts empty
- `add(name, phone)` — raises `ValueError` if the phone is not exactly 10 digits, or `KeyError` if the name already exists; saves to file on success
- `find(prefix)` — sorted list of names starting with `prefix` (case-insensitive)
- `remove(name)` — removes and saves; raises `KeyError` if missing

In [ ]:
class ContactBook:
    def __init__(self, filename):
        pass

    def add(self, name, phone):
        pass

    def find(self, prefix):
        pass

    def remove(self, name):
        pass

In [ ]:
book_file = path("book.json")
if os.path.exists(book_file): os.remove(book_file)
b = ContactBook(book_file)
b.add("Asha", "9876543210"); b.add("Arun", "9123456780"); b.add("Ravi", "9000000000")
check("find prefix (case-insensitive)", lambda: b.find("a") == ["Arun", "Asha"])
def _dup():
    try: b.add("Asha", "1111111111")
    except KeyError: return True
check("duplicate name -> KeyError", _dup)
def _bad_phone():
    try: b.add("Zed", "123")
    except ValueError: return True
check("bad phone -> ValueError", _bad_phone)
check("persisted to disk", lambda: ContactBook(book_file).contacts == {"Asha": "9876543210", "Arun": "9123456780", "Ravi": "9000000000"})
def _remove():
    b.remove("Ravi")
    try: b.remove("Ravi")
    except KeyError: return "Ravi" not in ContactBook(book_file).contacts
check("remove + KeyError when missing", _remove)

---
## 🏁 Next
≥ 90 %? Go to `06_OOP_challenges.ipynb`. **Bonus:** add a CSV export/import to `ContactBook`, and log every add/remove to a file with the `logging` module.